# Pizza Restaurant Sales Optimization

In [2]:
import pandas as pd
import glob

### Step 1: Load and merge the data

In [3]:
# 1. Get all CSV file paths in the 'Order' folder
csv_files = glob.glob("Order/Order_*.csv")

# 2. Load each file and combine them into a single DataFrame
df_list = [pd.read_csv(file) for file in csv_files]
df = pd.concat(df_list, ignore_index=True)

# Display a preview of the data and available columns
print("Data successfully loaded!")
df.head()

Data successfully loaded!


,Order,Date,Postcode,Total amount,Paid online,Pickup
0,R8WC83,4/1/2026 10:42,NaN,5.5,yes,yes
1,V87YQH,4/1/2026 17:45,NaN,24.5,yes,yes
2,MTRJQD,4/1/2026 17:49,9230.0,17.0,no,no
3,MQHVXH,4/1/2026 17:53,9520.0,30.1,yes,no
4,P77GVF,4/1/2026 18:06,9230.0,39.0,yes,no


### Step 2: Check and convert column formats

In [6]:
# Check initial data types
print(df.dtypes)

# --- Price Conversion ---
df['Total amount'] = pd.to_numeric(df['Total amount'], errors='coerce')

# --- Date Conversion ---
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

print("\nAfter conversion:")
print(df.dtypes)

Order            object
Date             object
Postcode        float64
Total amount    float64
Paid online      object
Pickup           object
dtype: object

After conversion:
Order                   object
Date            datetime64[ns]
Postcode               float64
Total amount           float64
Paid online             object
Pickup                  object
dtype: object


### Step 3: Clean empty rows or errors (Fixed)

In [7]:
# 1. Remove completely empty rows
df.dropna(how='all', inplace=True)

# 2. Remove rows where crucial information is missing (using your actual columns)
df.dropna(subset=['Total amount', 'Date'], inplace=True)

# 3. Filter out obvious register/cash errors (e.g., a total amount that is negative or equal to 0)
df = df[df['Total amount'] > 0]

print(f"Cleaning complete. {len(df)} rows remain ready for analysis.")

Cleaning complete. 2592 rows remain ready for analysis.


## Business Questions To Solve & Optimize 📊 

### 1.Revenue & Staffing Optimization (The Rush Hour)

In [8]:
import pandas as pd

# 1. Feature Engineering: Extracting Hour and Day Name from the Date column
df['Hour'] = df['Date'].dt.hour
df['Day_of_Week'] = df['Date'].dt.day_name()

print("--- RUSH HOUR ANALYSIS ---")

# 2. Top 3 Peak Days (by Order Volume and Revenue)
# Grouping by Day of Week to count total orders and sum total sales
daily_stats = df.groupby('Day_of_Week').agg(
    Total_Orders=('Order', 'count'),
    Total_Revenue=('Total amount', 'sum')
).sort_values(by='Total_Orders', ascending=False)

print("\nTop Days of the Week (Sorted by Order Volume):")
print(daily_stats.head(3))

# 3. Top 3 Peak Hours (by Order Volume and Revenue)
# Grouping by Hour to count total orders and sum total sales
hourly_stats = df.groupby('Hour').agg(
    Total_Orders=('Order', 'count'),
    Total_Revenue=('Total amount', 'sum')
).sort_values(by='Total_Orders', ascending=False)

print("\nTop 3 Peak Hours of the Day:")
print(hourly_stats.head(3))

# 4. Deep Dive: Cross-tabulation (Heatmap Prep)
# This creates a grid of Day vs Hour, which is perfect for Power BI visuals
rush_hour_matrix = pd.crosstab(df['Day_of_Week'], df['Hour'], values=df['Total amount'], aggfunc='sum').fillna(0)

--- RUSH HOUR ANALYSIS ---

Top Days of the Week (Sorted by Order Volume):
             Total_Orders  Total_Revenue
Day_of_Week                             
Sunday                445       11471.40
Saturday              437       11853.20
Thursday              393       10291.45

Top 3 Peak Hours of the Day:
      Total_Orders  Total_Revenue
Hour                             
18             417       11386.95
17             365       10432.90
19             349        9217.20


In [10]:
import pandas as pd

print("--- LOGISTICAL & DELIVERY ZONE OPTIMIZATION ---")

# 1. Total Revenue and Order Count per Postcode
postcode_revenue = df.groupby('Postcode').agg(
    Total_Revenue=('Total amount', 'sum'),
    Order_Count=('Order', 'count'),
    Average_Order_Value=('Total amount', 'mean')
).sort_values(by='Total_Revenue', ascending=False)

print("\nPostcode Performance (Sorted by Total Revenue):")
print(postcode_revenue)

# 2. Ratio of Delivery vs. Pickup per Postcode
delivery_pickup_ratio = pd.crosstab(df['Postcode'], df['Pickup'], normalize='index') * 100

# I used .rename() instead of forcing a 2-element list to avoid length mismatch crashes
delivery_pickup_ratio = delivery_pickup_ratio.rename(columns={'no': 'Delivery_%', 'yes': 'Pickup_%'})

print("\nDelivery vs. Pickup Ratio (%) per Postcode:")
# I sorted by 'Delivery_%' safely if it exists, otherwise print the raw table
if 'Delivery_%' in delivery_pickup_ratio.columns:
    print(delivery_pickup_ratio.sort_values(by='Delivery_%', ascending=False))
else:
    print(delivery_pickup_ratio)

--- LOGISTICAL & DELIVERY ZONE OPTIMIZATION ---

Postcode Performance (Sorted by Total Revenue):
          Total_Revenue  Order_Count  Average_Order_Value
Postcode                                                 
9230.0         33384.25         1414            23.609795
9270.0         11494.05          413            27.830630
9260.0          6490.80          240            27.045000
9860.0          3103.40           84            36.945238
9420.0          3008.20           75            40.109333
9290.0          2565.20           60            42.753333
9340.0          2064.20           57            36.214035
9520.0          1508.00           42            35.904762
9070.0           889.40           22            40.427273
9090.0           508.40           16            31.775000
9552.0           419.50           10            41.950000
9200.0           145.00            3            48.333333
9550.0            70.70            2            35.350000
9521.0            45.00          

In [12]:
import pandas as pd

print("--- PAYMENT METHOD & DIGITAL EXPERIENCE OPTIMIZATION ---")

# 1. Total Revenue and Percentage Split by Payment Method
payment_split = df.groupby('Paid online').agg(
    Total_Revenue=('Total amount', 'sum'),
    Order_Count=('Order', 'count')
)

total_business_revenue = df['Total amount'].sum()
payment_split['Revenue_%'] = (payment_split['Total_Revenue'] / total_business_revenue) * 100

print("\nRevenue Split by Payment Method:")
print(payment_split)

# 2. Average Spend Comparison (Panier Moyen)
average_spend = df.groupby('Paid online')['Total amount'].mean()

print("\nAverage Spend (Panier Moyen) by Payment Method:")
print(average_spend)

--- PAYMENT METHOD & DIGITAL EXPERIENCE OPTIMIZATION ---

Revenue Split by Payment Method:
             Total_Revenue  Order_Count  Revenue_%
Paid online                                       
no                  8928.9          378  13.079399
yes                59338.0         2214  86.920601

Average Spend (Panier Moyen) by Payment Method:
Paid online
no     23.621429
yes    26.801265
Name: Total amount, dtype: float64
